# Alkahest Qwen Text-Only Export

Builds a q4 WebGPU ONNX package from a merged Qwen3.5 Alkahest checkpoint. By default this is text-only and omits the fp16 vision encoder for faster browser cold loads; set `TEXT_EXPORT_INCLUDE_VISION=1` for the full text+image package.

In [ ]:
import os, platform, shutil
from pathlib import Path

print('python_platform=', platform.platform())
print('working_disk_free_gb=', round(shutil.disk_usage('/kaggle/working').free / 1024**3, 2))
print('source_repo_id=', os.environ.get('SOURCE_REPO_ID', 'thomasjvu/alkahest-2b-heretic-merged'))
print('target_repo_id=', os.environ.get('TARGET_REPO_ID', 'thomasjvu/alkahest-2b-heretic-q4-onnx-text'))

In [ ]:
import os, subprocess, sys, time

os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
secret_token = ''
for attempt in range(5):
    try:
        from kaggle_secrets import UserSecretsClient
        secret_token = UserSecretsClient().get_secret('HF_TOKEN')
        break
    except Exception as exc:
        print('hf_secret_attempt_failed=', attempt + 1, type(exc).__name__)
        time.sleep(3)
if secret_token and not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = secret_token
if secret_token and not os.environ.get('HUGGING_FACE_HUB_TOKEN'):
    os.environ['HUGGING_FACE_HUB_TOKEN'] = secret_token
print('hf_secret_loaded=', bool(secret_token))

packages = [
    'huggingface_hub[cli]>=1.5.0',
    'hf_transfer>=0.1.9',
    'safetensors>=0.7.0',
    'onnx>=1.19.0',
    'onnx-ir>=0.1.12',
    'onnxruntime>=1.23.0',
    'pyyaml>=6.0.0',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'pip'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *packages])
print('hf_token_present=', bool(os.environ.get('HF_TOKEN') or os.environ.get('HUGGING_FACE_HUB_TOKEN')))

In [ ]:
import os, subprocess
from pathlib import Path

REPO_URL = os.environ.get('HERETIC_TO_ONNX_REPO', 'https://github.com/alkahest-ai/heretic-to-onnx.git')
REPO_REF = os.environ.get('HERETIC_TO_ONNX_REF', 'codex/kaggle-heretic-2b-run')
REPO_DIR = Path('/kaggle/working/heretic-to-onnx')

if REPO_DIR.exists():
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'fetch', 'origin', REPO_REF])
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'checkout', REPO_REF])
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'])
else:
    subprocess.check_call(['git', 'clone', '--branch', REPO_REF, '--depth', '1', REPO_URL, str(REPO_DIR)])

print('repo=', REPO_DIR)
print('head=', subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', '--short', 'HEAD'], text=True).strip())

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO_DIR = Path('/kaggle/working/heretic-to-onnx')
WORK_DIR = Path(os.environ.get('TEXT_EXPORT_WORK_DIR', '/kaggle/working/alkahest-qwen-text-export'))
cmd = [
    sys.executable,
    str(REPO_DIR / 'scripts/kaggle_alkahest_qwen_text_export.py'),
    '--source-repo-id', os.environ.get('SOURCE_REPO_ID', 'thomasjvu/alkahest-2b-heretic-merged'),
    '--template-model-id', os.environ.get('TEMPLATE_MODEL_ID', 'onnx-community/Qwen3.5-2B-ONNX-OPT'),
    '--base-model-id', os.environ.get('BASE_MODEL_ID', 'Qwen/Qwen3.5-2B'),
    '--target-repo-id', os.environ.get('TARGET_REPO_ID', 'thomasjvu/alkahest-2b-heretic-q4-onnx-text'),
    '--work-dir', str(WORK_DIR),
]
if os.environ.get('TEXT_EXPORT_NO_UPLOAD') == '1':
    cmd.append('--no-upload')
elif not (os.environ.get('HF_TOKEN') or os.environ.get('HUGGING_FACE_HUB_TOKEN')):
    cmd.append('--no-upload')
if os.environ.get('TEXT_EXPORT_PUBLIC') == '1':
    cmd.append('--no-private')
if os.environ.get('TEXT_EXPORT_INCLUDE_VISION') == '1':
    cmd.append('--include-vision')
if os.environ.get('TEXT_EXPORT_NO_RUNTIME_SMOKE') == '1':
    cmd.append('--no-runtime-smoke')
print('running=', ' '.join(cmd))
subprocess.check_call(cmd)

In [ ]:
from pathlib import Path
import json, os

target = os.environ.get('TARGET_REPO_ID', 'thomasjvu/alkahest-2b-heretic-q4-onnx-text').replace('/', '__')
report_path = Path(os.environ.get('TEXT_EXPORT_WORK_DIR', '/kaggle/working/alkahest-qwen-text-export')) / 'package' / target / 'text-export-report.json'
print('report_path=', report_path)
if report_path.exists():
    report = json.loads(report_path.read_text())
    print(json.dumps({
        'ok': report.get('ok'),
        'target_repo_id': report.get('target_repo_id'),
        'package_size_gb': report.get('package_size_gb'),
        'validation_ok': report.get('validation', {}).get('ok'),
        'upload': report.get('upload'),
    }, indent=2))
else:
    raise FileNotFoundError(report_path)